# PlateVision LPR: train the YOLOv11 plate detector on Google Colab

Runs on Google's free GPU, so it works from a phone browser or a low-power laptop.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

You need your labelled dataset as a zip in Google Drive, for example `MyDrive/lpr/raw_plates.zip`,
containing images and YOLO `.txt` labels (class `0 = license_plate`), either side by side or in
`images/` and `labels/` folders.

In [ ]:
# 1. Connect Google Drive (your dataset and trained weights live there)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Get the code and install the training dependencies
!git clone -b claude/license-plate-detection-ocr-9aaupw https://github.com/amnaanamds-cmyk/Intelligent-license-plate-detection-and-recognition-system-.git lpr
%cd /content/lpr
!pip install -q ultralytics lap

In [ ]:
# 3. Unpack and split the dataset (70% train / 15% val / 15% test)
DATASET_ZIP = '/content/drive/MyDrive/lpr/raw_plates.zip'   # <-- change if needed
!mkdir -p /content/raw && unzip -q -o "$DATASET_ZIP" -d /content/raw
!python scripts/split_dataset.py --src /content/raw --dst datasets/plates --val 0.15 --test 0.15

In [ ]:
# 4. Train. yolo11s is a good balance; use yolo11n for weak deployment hardware.
!python scripts/train.py --data configs/data.yaml --model yolo11s.pt --epochs 100 --batch 16 --device 0

In [ ]:
# 5. Evaluate on the held-out test split: precision, recall, F1, mAP@50 (Table 1 of the paper)
!python scripts/evaluate.py detect --weights weights/plate_yolo11.pt --split test --device 0

In [ ]:
# 6. Training curves (loss, precision/recall, PR curve)
from IPython.display import Image, display
for f in ['results.png', 'PR_curve.png', 'confusion_matrix.png']:
    try:
        display(Image(filename=f'runs/detect/plate_yolo11/{f}', width=900))
    except FileNotFoundError:
        pass

In [ ]:
# 7. Save the trained model to Drive (and an ONNX copy for fast CPU inference)
!python scripts/export.py --weights weights/plate_yolo11.pt --format onnx
!mkdir -p /content/drive/MyDrive/lpr/weights
!cp weights/plate_yolo11.pt weights/plate_yolo11.onnx /content/drive/MyDrive/lpr/weights/
print('Saved to MyDrive/lpr/weights - copy plate_yolo11.pt into the weights/ folder of your installation.')

## Optional: quick detection test on one image
Upload a photo with the file browser on the left, then set its path below.

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
res = YOLO('weights/plate_yolo11.pt').predict('/content/car.jpg', conf=0.35, save=True)
display(Image(filename=res[0].save_dir + '/car.jpg', width=700))